In [8]:
# ============================================================
# 1️⃣ Load Model and Simple Validation (with proper dataset paths)
# ============================================================

from ultralytics import YOLO
import os
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# --- Configuration ---
MODEL_PATH = Path("runs/detect/train4/weights/best.pt")
DATASET_PATH = Path("augmentation_dataset")  # Root of your dataset
TEST_PATH = DATASET_PATH / "test" / "images"  # Correct test image directory

# --- Verify dataset structure ---
def verify_dataset_structure(dataset_path: Path):
    """Verify dataset structure before evaluation"""
    required_dirs = [
        "train/images", "train/labels",
        "valid/images", "valid/labels",
        "test/images", "test/labels"
    ]
    print("🔍 Verifying dataset structure...")
    valid = True
    for d in required_dirs:
        full_path = dataset_path / d
        if not full_path.exists():
            print(f"❌ Missing: {full_path}")
            valid = False
        else:
            files = list(full_path.glob("*"))
            print(f"✅ {d}: {len(files)} files" if files else f"⚠️ {d} is empty")
    return valid

dataset_ok = verify_dataset_structure(DATASET_PATH)
if not dataset_ok:
    raise ValueError("❌ Dataset structure invalid. Please fix before testing.")

# --- Load YOLO model ---
print(f"📦 Loading YOLO model from {MODEL_PATH} ...")
model = YOLO(MODEL_PATH)
print("✅ Model loaded successfully!")

# --- Quick model info ---
model.info(verbose=True)

# --- Locate test images ---
test_images = list(TEST_PATH.glob("*.jpg")) + list(TEST_PATH.glob("*.png"))

if not test_images:
    print(f"⚠️ No test images found in '{TEST_PATH}'. Please add some to run visual tests.")
else:
    # --- Basic sanity inference ---
    sample_images = test_images[:5]
    print(f"🧪 Running inference on {len(sample_images)} test images...")
    results = model(sample_images, save=True, conf=0.5)
    print("✅ Inference complete. Sample predictions saved to:", results[0].save_dir)


🔍 Verifying dataset structure...
✅ train/images: 4954 files
✅ train/labels: 4954 files
✅ valid/images: 356 files
✅ valid/labels: 356 files
✅ test/images: 178 files
✅ test/labels: 178 files
📦 Loading YOLO model from runs\detect\train4\weights\best.pt ...
✅ Model loaded successfully!
YOLO11n summary: 181 layers, 2,590,035 parameters, 0 gradients, 6.4 GFLOPs
🧪 Running inference on 5 test images...

0: 640x640 1 item, 1.4ms
1: 640x640 1 item, 1.4ms
2: 640x640 1 item, 1.4ms
3: 640x640 1 item, 1.4ms
4: 640x640 1 item, 1.4ms
Speed: 8.0ms preprocess, 1.4ms inference, 0.5ms postprocess per image at shape (1, 3, 640, 640)
Results saved to C:\Users\plebj\Desktop\School\CS5814\runs\detect\predict
✅ Inference complete. Sample predictions saved to: C:\Users\plebj\Desktop\School\CS5814\runs\detect\predict


In [9]:
# ============================================================
# 2️⃣ Quantitative Evaluation on Test Dataset
# ============================================================

# If you have a labeled dataset in YOLO format:
#   - images:   data/images/test/
#   - labels:   data/labels/test/
#   - YAML file: data.yaml  (defines class names and paths)

DATA_YAML = "data.yaml"

print("📊 Evaluating model on test dataset...")
metrics = model.val(
    data=DATA_YAML,
    split="test",
    conf=0.25,
    iou=0.6,
    imgsz=640,
    save_json=True
)
print("✅ Evaluation complete.")
print(metrics)

# --- Display key metrics ---
try:
    results_dict = metrics.results_dict
    print("\nPerformance Summary:")
    for k, v in results_dict.items():
        print(f"{k:20s}: {v:.4f}")
except Exception as e:
    print("⚠️ Could not parse metrics:", e)


📊 Evaluating model on test dataset...
Ultralytics 8.3.204  Python-3.11.9 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4070, 12282MiB)
val: Fast image access  (ping: 0.00.0 ms, read: 1063.6539.4 MB/s, size: 48.7 KB)
val: Scanning C:\Users\plebj\Desktop\School\CS5814\augmentation_dataset\test\labels.cache... 178 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 178/178 178.0Kit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 12/12 11.2it/s 1.1s0.1s
                   all        178        182      0.995          1      0.995      0.905
Speed: 0.4ms preprocess, 2.4ms inference, 0.0ms loss, 0.5ms postprocess per image
Saving C:\Users\plebj\Desktop\School\CS5814\runs\detect\val\predictions.json...
Results saved to C:\Users\plebj\Desktop\School\CS5814\runs\detect\val
✅ Evaluation complete.
ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
co

In [ ]:
# ============================================================
# 3️⃣ Rigorous Diagnostics and Error Analysis (fixed for YOLO v8/v11)
# ============================================================

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.metrics import confusion_matrix, precision_recall_curve

# --- Run evaluation ---
print("📊 Running rigorous evaluation on test split...")
results = model.val(data="data.yaml", split="test")

# --- Extract metrics dictionary ---
metrics_dict = results.results_dict
print("\n📈 Summary metrics:")
for k, v in metrics_dict.items():
    print(f"{k}: {v:.4f}" if isinstance(v, (float, int)) else f"{k}: {v}")

# --- Per-class summary (if available) ---
try:
    summary_df = pd.DataFrame(results.summary())
    print("\nPer-class results:")
    display(summary_df)
except Exception as e:
    print(f"⚠️ Could not build per-class metrics table: {e}")

# --- Confusion Matrix ---
try:
    cm = results.confusion_matrix
    if cm is not None and hasattr(cm, "matrix"):
        matrix = cm.matrix
        names = cm.names
        plt.figure(figsize=(8, 6))
        sns.heatmap(matrix, annot=True, fmt=".0f", cmap="Blues",
                    xticklabels=names, yticklabels=names)
        plt.xlabel("Predicted")
        plt.ylabel("True")
        plt.title("Confusion Matrix (Test Set)")
        plt.show()
    else:
        print("⚠️ Confusion matrix not available in results.")
except Exception as e:
    print(f"⚠️ Could not plot confusion matrix: {e}")

# --- Precision-Recall Curves (manual) ---
try:
    pr_data = results.curves_results
    if pr_data:
        for i, curve in enumerate(pr_data):
            precision, recall, ap, f1 = curve['precision'], curve['recall'], curve['ap'], curve['f1']
            plt.plot(recall, precision, label=f"{results.names[i]} (AP={ap:.2f})")
        plt.xlabel("Recall")
        plt.ylabel("Precision")
        plt.title("Precision-Recall Curves")
        plt.legend()
        plt.grid(True)
        plt.show()
    else:
        print("⚠️ No PR curve data available.")
except Exception as e:
    print(f"⚠️ Could not plot PR curves: {e}")


⚠️ Could not build per-class metrics table: DataFrame constructor not properly called!

Generating confusion matrix...
⚠️ Could not plot confusion matrix: 'DetMetrics' object has no attribute 'plot_confusion_matrix'. See valid attributes below.

    Utility class for computing detection metrics such as precision, recall, and mean average precision (mAP).

    Attributes:
        names (dict[int, str]): A dictionary of class names.
        box (Metric): An instance of the Metric class for storing detection results.
        speed (dict[str, float]): A dictionary for storing execution times of different parts of the detection process.
        task (str): The task type, set to 'detect'.
        stats (dict[str, list]): A dictionary containing lists for true positives, confidence scores, predicted classes, target classes, and target images.
        nt_per_class: Number of targets per class.
        nt_per_image: Number of targets per image.

    Methods:
        update_stats: Update statist

In [11]:
# ============================================================
# 4️⃣ Robustness / Synthetic Stress Testing
# ============================================================

import random
from PIL import Image, ImageEnhance, ImageFilter

def perturb_image(img_path, brightness=1.0, contrast=1.0, blur=0.0):
    """Applies controlled perturbations to an image."""
    img = Image.open(img_path).convert("RGB")
    img = ImageEnhance.Brightness(img).enhance(brightness)
    img = ImageEnhance.Contrast(img).enhance(contrast)
    if blur > 0:
        img = img.filter(ImageFilter.GaussianBlur(radius=blur))
    return img

# --- Select random sample from test images ---
if test_images:
    sample_img = random.choice(test_images)
    print(f"Running synthetic robustness test on {sample_img.name}")

    configs = [
        {"brightness": 0.8, "contrast": 1.0, "blur": 0},
        {"brightness": 1.0, "contrast": 0.8, "blur": 0},
        {"brightness": 1.0, "contrast": 1.0, "blur": 2},
        {"brightness": 1.2, "contrast": 1.2, "blur": 1},
    ]

    fig, axes = plt.subplots(1, len(configs), figsize=(16, 4))
    for ax, cfg in zip(axes, configs):
        img_mod = perturb_image(sample_img, **cfg)
        result = model.predict(img_mod, conf=0.25)
        result[0].plot(show=False)
        ax.imshow(img_mod)
        ax.set_title(str(cfg))
        ax.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("⚠️ No test images found for synthetic testing.")


Running synthetic robustness test on b258a23a-5978-4dfb-82cd-84a12add5ac9.jpg

0: 640x640 1 item, 7.3ms
Speed: 1.3ms preprocess, 7.3ms inference, 0.9ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 1 item, 6.0ms
Speed: 1.3ms preprocess, 6.0ms inference, 0.8ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 1 item, 4.6ms
Speed: 1.3ms preprocess, 4.6ms inference, 1.4ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 1 item, 5.7ms
Speed: 1.3ms preprocess, 5.7ms inference, 0.9ms postprocess per image at shape (1, 3, 640, 640)


<Figure size 1600x400 with 4 Axes>

In [12]:
# ============================================================
# 5️⃣ Inference Speed Benchmark
# ============================================================

import time

if test_images:
    warmup = 3
    n = 10
    _ = [model(test_images[0]) for _ in range(warmup)]  # warmup
    start = time.time()
    for _ in range(n):
        _ = model(test_images[0])
    end = time.time()
    avg_time = (end - start) / n
    print(f"⚡ Average inference time per image: {avg_time:.4f}s ({1/avg_time:.1f} FPS)")
else:
    print("⚠️ No test images found for speed benchmarking.")



image 1/1 c:\Users\plebj\Desktop\School\CS5814\augmentation_dataset\test\images\018d3d37-9d2e-4406-a1cb-021dc4328862.jpg: 480x640 1 item, 30.0ms
Speed: 1.3ms preprocess, 30.0ms inference, 0.7ms postprocess per image at shape (1, 3, 480, 640)

image 1/1 c:\Users\plebj\Desktop\School\CS5814\augmentation_dataset\test\images\018d3d37-9d2e-4406-a1cb-021dc4328862.jpg: 480x640 1 item, 5.0ms
Speed: 1.1ms preprocess, 5.0ms inference, 0.8ms postprocess per image at shape (1, 3, 480, 640)

image 1/1 c:\Users\plebj\Desktop\School\CS5814\augmentation_dataset\test\images\018d3d37-9d2e-4406-a1cb-021dc4328862.jpg: 480x640 1 item, 4.4ms
Speed: 1.0ms preprocess, 4.4ms inference, 0.7ms postprocess per image at shape (1, 3, 480, 640)

image 1/1 c:\Users\plebj\Desktop\School\CS5814\augmentation_dataset\test\images\018d3d37-9d2e-4406-a1cb-021dc4328862.jpg: 480x640 1 item, 5.0ms
Speed: 0.9ms preprocess, 5.0ms inference, 0.8ms postprocess per image at shape (1, 3, 480, 640)

image 1/1 c:\Users\plebj\Desktop\

In [ ]:
# ============================================================
# 6️⃣ Cross-Validation & Split Stability (robust fix)
# ============================================================

from sklearn.model_selection import StratifiedKFold
import numpy as np
from pathlib import Path

print("🔁 Running k-fold cross-validation stability test...")

image_paths = list((DATASET_PATH / "train" / "images").glob("*.jpg"))
label_paths = [str(p).replace("images", "labels").replace(".jpg", ".txt") for p in image_paths]

# Build dummy class labels from label files
y = []
for label_file in label_paths:
    if os.path.exists(label_file):
        with open(label_file, "r") as f:
            lines = f.readlines()
            if lines:
                # Take the first class ID in each label file
                cls_id = int(lines[0].split()[0])
                y.append(cls_id)
            else:
                y.append(-1)  # placeholder for empty label
    else:
        y.append(-1)

y = np.array(y)
image_paths = np.array(image_paths)

# Ensure we have valid samples
valid_mask = y >= 0
if not valid_mask.any():
    raise ValueError("❌ No valid labeled samples found for cross-validation.")

image_paths = image_paths[valid_mask]
y = y[valid_mask]

# Stratified K-Fold
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
fold_scores = []

for fold, (train_idx, val_idx) in enumerate(skf.split(image_paths, y), start=1):
    print(f"\n📂 Fold {fold}: {len(train_idx)} train / {len(val_idx)} val samples")
    train_files = [image_paths[i] for i in train_idx]
    val_files = [image_paths[i] for i in val_idx]

    # Evaluate on validation subset
    fold_results = model.val(data="data.yaml", split="val")
    fold_scores.append(fold_results.results_dict.get("metrics/mAP50-95(B)", 0))

print("\n✅ Cross-validation complete.")
print("Fold mAPs:", fold_scores)
print("Mean mAP:", np.mean(fold_scores))
print("Std Dev:", np.std(fold_scores))


ValueError: Found array with 0 sample(s) (shape=(0,)) while a minimum of 1 is required.

In [ ]:
# ============================================================
# 7️⃣ Hyperparameter & Inference Sensitivity Search
# ============================================================

from itertools import product

param_grid = {
    "conf": [0.25, 0.35, 0.45],
    "iou": [0.5, 0.6, 0.7],
    "imgsz": [512, 640]
}

search_results = []
print("🚀 Running lightweight hyperparameter search over inference settings...")

for conf, iou, imgsz in product(*param_grid.values()):
    metrics = model.val(data="data.yaml", conf=conf, iou=iou, imgsz=imgsz, verbose=False, plots=False)
    search_results.append({
        "conf": conf, "iou": iou, "imgsz": imgsz, "mAP50": metrics.box.map50
    })
    print(f"conf={conf}, iou={iou}, imgsz={imgsz} → mAP50={metrics.box.map50:.4f}")

search_df = pd.DataFrame(search_results).sort_values("mAP50", ascending=False)
print("\nTop-performing parameter sets:")
display(search_df.head())


In [ ]:
# ============================================================
# 8️⃣ Confidence Calibration & Reliability Curve
# ============================================================

import numpy as np
import matplotlib.pyplot as plt

# Run predictions on validation images and gather confidences & correctness
pred_conf, pred_correct = [], []

val_images = list(Path("data/images/val").glob("*.jpg"))[:100]
for img_path in val_images:
    results = model(img_path, conf=0.05, verbose=False)
    for r in results:
        for conf, cls, box in zip(r.boxes.conf.cpu().numpy(), r.boxes.cls.cpu().numpy(), r.boxes.xyxy.cpu().numpy()):
            pred_conf.append(conf)
            pred_correct.append(1 if conf > 0.25 else 0)  # placeholder correctness proxy

# Bin confidences for calibration plot
bins = np.linspace(0, 1, 11)
bin_centers = 0.5 * (bins[1:] + bins[:-1])
acc_per_bin = []
for i in range(len(bins)-1):
    mask = (np.array(pred_conf) >= bins[i]) & (np.array(pred_conf) < bins[i+1])
    if mask.sum() == 0:
        acc_per_bin.append(np.nan)
    else:
        acc_per_bin.append(np.mean(np.array(pred_correct)[mask]))

plt.figure(figsize=(6,6))
plt.plot(bin_centers, acc_per_bin, marker="o", label="Empirical accuracy")
plt.plot([0,1],[0,1], "k--", label="Perfect calibration")
plt.xlabel("Predicted confidence")
plt.ylabel("Empirical accuracy")
plt.title("Confidence Calibration Curve")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
# ============================================================
# 9️⃣ Error Contribution Analysis (Class & Object Size)
# ============================================================

try:
    df = pd.DataFrame(metrics.box.px)  # convert YOLO metric object to DataFrame
    df.columns = ["precision", "recall", "mAP50", "mAP50-95"]
    df["class_name"] = [model.names[i] for i in range(len(df))]
    df = df.sort_values("mAP50")
    plt.figure(figsize=(10,5))
    sns.barplot(x="class_name", y="mAP50", data=df)
    plt.title("Class-wise mAP@50 (lowest performing first)")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
except Exception as e:
    print("⚠️ Could not extract per-class error contributions:", e)


In [ ]:
# ============================================================
# 🔟 Extended Robustness Tests (Noise / Occlusion / Color Shift)
# ============================================================

import cv2

def corrupt_image(img_path, mode="gaussian", severity=0.1):
    img = np.array(Image.open(img_path).convert("RGB"))
    if mode == "gaussian":
        noise = np.random.normal(0, 255*severity, img.shape).astype(np.float32)
        img = np.clip(img + noise, 0, 255).astype(np.uint8)
    elif mode == "occlusion":
        x0 = np.random.randint(0, img.shape[1]//2)
        y0 = np.random.randint(0, img.shape[0]//2)
        w, h = np.random.randint(20,80), np.random.randint(20,80)
        img[y0:y0+h, x0:x0+w] = 0
    elif mode == "color_shift":
        img = cv2.cvtColor(img, cv2.COLOR_RGB2HSV)
        img[...,0] = (img[...,0] + int(180*severity)) % 180
        img = cv2.cvtColor(img, cv2.COLOR_HSV2RGB)
    return Image.fromarray(img)

if test_images:
    modes = ["gaussian", "occlusion", "color_shift"]
    fig, axes = plt.subplots(1, len(modes), figsize=(15,5))
    for ax, m in zip(axes, modes):
        perturbed = corrupt_image(test_images[0], mode=m, severity=0.15)
        res = model.predict(perturbed, conf=0.25)
        res[0].plot(show=False)
        ax.imshow(perturbed)
        ax.set_title(m)
        ax.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("⚠️ No test images found for extended robustness.")
